In [ ]:
!pip install qiskit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 5.1 MB/s eta 0:00:00


In [ ]:
from qiskit.circuit.library import QAOAAnsatz
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit.visualization import plot_histogram
from qiskit.primitives import StatevectorEstimator, StatevectorSampler
from scipy.optimize import minimize
import numpy as np

Activity 1

In [ ]:
edges = [(0,1), (0,2), (1,2), (1,3), (2,4), (3,4)]

Hc = SparsePauliOp.from_sparse_list(
    [("ZZ", [i, j], 1.0) for (i, j) in edges],
    num_qubits=5
)
print(Hc.to_matrix(sparse=True))

<Compressed Sparse Row sparse matrix of dtype 'complex128'
	with 32 stored elements and shape (32, 32)>
  Coords	Values
  (0, 0)	(6+0j)
  (1, 1)	(2+0j)
  (2, 2)	0j
  (3, 3)	0j
  (4, 4)	0j
  (5, 5)	0j
  (6, 6)	(-2+0j)
  (7, 7)	(2+0j)
  (8, 8)	(2+0j)
  (9, 9)	(-2+0j)
  (10, 10)	0j
  (11, 11)	0j
  (12, 12)	(-4+0j)
  (13, 13)	(-4+0j)
  (14, 14)	(-2+0j)
  (15, 15)	(2+0j)
  (16, 16)	(2+0j)
  (17, 17)	(-2+0j)
  (18, 18)	(-4+0j)
  (19, 19)	(-4+0j)
  (20, 20)	0j
  (21, 21)	0j
  (22, 22)	(-2+0j)
  (23, 23)	(2+0j)
  (24, 24)	(2+0j)
  (25, 25)	(-2+0j)
  (26, 26)	0j
  (27, 27)	0j
  (28, 28)	0j
  (29, 29)	0j
  (30, 30)	(2+0j)
  (31, 31)	(6+0j)


Activity 2

In [ ]:
from qiskit import QuantumCircuit


estimator = StatevectorEstimator()


min = 10000
index = 0
for i in range(32):
  state = Statevector.from_int(i, dims=32)
  #print(state)

  qc = QuantumCircuit(5)
  qc.initialize(state, range(5))

  job = estimator.run([(qc, Hc)])
  expectation_value_empirical = job.result()[0].data.evs
  if (expectation_value_empirical < min):
    min = expectation_value_empirical
    index = i


print("value:", min, "Index:", index)




value: -4.0 Index: 12


Ici on trouve que un de nos minimums vaut -4 (on pouvait déja le voir à. l'activité précédente) et qu'il se trouve à notre état de valeur 12  de la diagonale de notre matrice. Avec comme valeur -4 on peut en déduire qu'on coupe 5 fois notre graph ici.

Activity 3


L'opérateur $Z \otimes Z$ vérifie $(Z \otimes Z)^2 = Z^2 \otimes Z^2 = I$, car $Z^2 = I$.

Pour tout opérateur $A$ tel que $A^2 = I$, on a la formule:

$$e^{-i\gamma A} = \cos(\gamma)\, I - i\sin(\gamma)\, A$$

Donc :

$$U_C(\gamma) = \cos(\gamma)\, I - i\sin(\gamma)\, (Z \otimes Z)$$

Dans la base computationelle, la matrice de $Z \otimes Z$ est diagonale, ses éléments étant les valeurs propres $z_i z_j$ :

(1, -1, -1, 1)  sur la diagonale

Chaque élément diagonal vaut alors $\cos\gamma \mp i\sin\gamma = e^{\mp i\gamma}$ en remplacant dans la matrice ce qui nous ramène à la matrice dans l'encadré.



Activity 4

In [ ]:
AnsatzCircuit = QAOAAnsatz(Hc, reps=2)
AnsatzCircuit.decompose(reps=3).draw()

┌────────────┐                                                »
q_0: ┤ U(π/2,0,π) ├──■──────────────────■────■──────────────────■──»
     ├────────────┤┌─┴─┐┌────────────┐┌─┴─┐  │                  │  »
q_1: ┤ U(π/2,0,π) ├┤ X ├┤ Rz(2*γ[0]) ├┤ X ├──┼──────────────────┼──»
     ├────────────┤└───┘└────────────┘└───┘┌─┴─┐┌────────────┐┌─┴─┐»
q_2: ┤ U(π/2,0,π) ├────────────────────────┤ X ├┤ Rz(2*γ[0]) ├┤ X ├»
     ├────────────┤                        └───┘└────────────┘└───┘»
q_3: ┤ U(π/2,0,π) ├────────────────────────────────────────────────»
     ├────────────┤                                                »
q_4: ┤ U(π/2,0,π) ├────────────────────────────────────────────────»
     └────────────┘                                                »
«     ┌─────────────┐                                                »
«q_0: ┤ R(2*β[0],0) ├────────────────────────────────────────────────»
«     └─────────────┘                                                »
«q_1: ───────■───────────────────────■────■───────────────────────■──»
«          ┌─┴─┐     ┌────────────┐┌─┴─┐  │                       │  »
«q_2: ─────┤ X ├─────┤ Rz(2*γ[0]) ├┤ X ├──┼────■──────────────────┼──»
«          └───┘     └────────────┘└───┘┌─┴─┐  │  ┌────────────┐┌─┴─┐»
«q_3: ──────────────────────────────────┤ X ├──┼──┤ Rz(2*γ[0]) ├┤ X ├»
«                                       └───┘┌─┴─┐├────────────┤└───┘»
«q_4: ───────────────────────────────────────┤ X ├┤ Rz(2*γ[0]) ├─────»
«                                            └───┘└────────────┘     »
«                                                                     »
«q_0: ──────────────────────■───────────────────────■─────────■───────»
«     ┌─────────────┐     ┌─┴─┐     ┌────────────┐┌─┴─┐       │       »
«q_1: ┤ R(2*β[0],0) ├─────┤ X ├─────┤ Rz(2*γ[1]) ├┤ X ├───────┼───────»
«     └─────────────┘┌────┴───┴────┐└────────────┘└───┘     ┌─┴─┐     »
«q_2: ───────■───────┤ R(2*β[0],0) ├────────────────────────┤ X ├─────»
«            │       └─────────────┘                   ┌────┴───┴────┐»
«q_3: ───────┼──────────────■───────────────────────■──┤ R(2*β[0],0) ├»
«          ┌─┴─┐          ┌─┴─┐     ┌────────────┐┌─┴─┐├─────────────┤»
«q_4: ─────┤ X ├──────────┤ X ├─────┤ Rz(2*γ[0]) ├┤ X ├┤ R(2*β[0],0) ├»
«          └───┘          └───┘     └────────────┘└───┘└─────────────┘»
«                        ┌─────────────┐                             »
«q_0: ────────────────■──┤ R(2*β[1],0) ├─────────────────────────────»
«                     │  └─────────────┘                             »
«q_1: ────────────────┼─────────■───────────────────────■────■───────»
«     ┌────────────┐┌─┴─┐     ┌─┴─┐     ┌────────────┐┌─┴─┐  │       »
«q_2: ┤ Rz(2*γ[1]) ├┤ X ├─────┤ X ├─────┤ Rz(2*γ[1]) ├┤ X ├──┼────■──»
«     └────────────┘└───┘     └───┘     └────────────┘└───┘┌─┴─┐  │  »
«q_3: ─────────────────────────────────────────────────────┤ X ├──┼──»
«                                                          └───┘┌─┴─┐»
«q_4: ──────────────────────────────────────────────────────────┤ X ├»
«                                                               └───┘»
«                                                                         »
«q_0: ────────────────────────────────────────────────────────────────────»
«                        ┌─────────────┐                                  »
«q_1: ────────────────■──┤ R(2*β[1],0) ├──────────────────────────────────»
«                     │  └─────────────┘┌─────────────┐                   »
«q_2: ────────────────┼─────────■───────┤ R(2*β[1],0) ├───────────────────»
«     ┌────────────┐┌─┴─┐       │       └─────────────┘                   »
«q_3: ┤ Rz(2*γ[1]) ├┤ X ├───────┼──────────────■───────────────────────■──»
«     ├────────────┤└───┘     ┌─┴─┐          ┌─┴─┐     ┌────────────┐┌─┴─┐»
«q_4: ┤ Rz(2*γ[1]) ├──────────┤ X ├──────────┤ X ├─────┤ Rz(2*γ[1]) ├┤ X ├»
«     └────────────┘          └───┘          └───┘     └────────────┘└───┘»
«                    
«q_0: ───────────────
«           

On a instancié un circuit anstatz de taille 2. Il est possible de changer l'affichage avec l'hyperparamêtre reps de la fonction decompose afin de détailler plus ou moins le circuit.

Activity 5

In [ ]:
def cost(parameters, ansatz_circuit: QAOAAnsatz, hc: SparsePauliOp):

  estimator = StatevectorEstimator()

  job = estimator.run([(ansatz_circuit, hc, parameters)])

  return job.result()[0].data.evs

#print(cost(AnsatzCircuit, Hc, [0.5, 0.2, 0.5, 0.2]))


Ici notre fonction cost effectue la moyenne ⟨ψp(γ,β)| HC |ψp(γ,β)⟩. Avec les vecteurs γ,β qui seront initialisés aléatoirement puis optimisés par la suite.

Activity 6

In [ ]:
rng = np.random.default_rng()
params = rng.uniform(0, 2 * np.pi, size=4)

print(params)

[0.35227496 1.40831947 0.59614453 5.50487997]


On utilise l'outil de génération de nombre aléatoire de numpy et on lui spécifie une taille de 4 ici car notre circuit à été initialisé avec un nombre de reps à 2.

Activity 7

In [ ]:
min = minimize(cost, params, (AnsatzCircuit, Hc), "COBYLA")

print(min)

/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/usr/local/lib/python3.12/dist-packages/scipy/sparse/_index.py:168: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


 message: Return from COBYLA because the trust region radius reaches its lower bound.
 success: True
  status: 0
     fun: -3.2468594631266576
       x: [ 1.053e+00  1.262e+00  2.925e-01  6.798e+00]
    nfev: 71
   maxcv: 0.0


Ici on peut appercevoir tout un tas de champs retourné par la fonction minimize. Ici on s'interesse au champ x qui est la liste des nos paramêtre optimiser.

Activity 8

In [ ]:
opti_parameters = [1.053e+00,  1.262e+00,  2.925e-01,  6.798e+00]


AnsatzCircuitOpti = QAOAAnsatz(Hc, reps=2)
AnsatzCircuitOpti.assign_parameters(opti_parameters, inplace=True)

AnsatzCircuit.measure_all()

job = StatevectorSampler().run([AnsatzCircuitOpti], shots=1024)
result = job.result()

data = result[0].data
#count = data.c.get_counts()

print(data)

/usr/local/lib/python3.12/dist-packages/qiskit/primitives/statevector_sampler.py:163: UserWarning: One of your circuits has no output classical registers and so the result will be empty. Did you mean to add measurement instructions?
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/usr/local/lib/python3.12/dist-packages/scipy/sparse/_index.py:168: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


DataBin()


Activity 9

Si on reprend le même problème mais cette fois-ci avec un poids sur les arrêtes alors pour un même nombre de coupe le poids final de couts ne sera potentiellement pas le même. Il faut que le score total de coupe (somme des coupes multiplier par leur poids sois le plus élevé possible)

Ce qui change pour notre application de QAOA est qu'il faut rajouter des coefficients initiaux pour chacun de nos vertex de notre graphe pour que notre Hamiltonian cost soit en adéquation avec le problême.

